In [1]:
# 1. Force remove the wrong 'edgar' package and alternatives
%pip uninstall edgar python-edgar sec-api -y

# 2. Install the correct package (it must be 'edgartools')
%pip install edgartools

Note: you may need to restart the kernel to use updated packages.


Note: you may need to restart the kernel to use updated packages.


In [2]:
# %pip uninstall edgar -y
# ! pip install edgar

# ! pip install sec-api

from edgar import *
set_identity("Jason Rodriguez jabob2002@gmail.com")
import pandas as pd
import numpy as np

c:\Users\jabob\.conda\conda\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\jabob\.conda\conda\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [3]:
#  test 


amex = Company("AXP")


cik = amex.cik

files = amex.get_filings(form="10-Q").head(10)

print(cik)

4962


In [4]:
apple = Company("AAPL")


apple_cik = amex.cik

files = apple.get_filings(form="10-Q").head(10)

print(cik)

4962


In [5]:
import pandas as pd

df = files.to_pandas()

df.columns = df.columns.str.lower()

time_stamps_columns = ['filing_date', 'reportdate']


quarter_col = 'reportdate'

df[quarter_col] = pd.to_datetime(df[quarter_col])
df['quarter_year'] = (
    df['reportdate'].dt.year.astype(str) 
    + "'Q" 
    + df['reportdate'].dt.quarter.astype(str)
)

df

,accession_number,filing_date,reportdate,acceptancedatetime,act,form,filenumber,items,size,isxbrl,isinlinexbrl,primarydocument,primarydocdescription,quarter_year
0,0000320193-26-000013,2026-05-01,2026-03-28,2026-05-01 10:01:00+00:00,34,10-Q,001-36743,,5809851,1,1,aapl-20260328.htm,10-Q,2026'Q1
1,0000320193-26-000006,2026-01-30,2025-12-27,2026-01-30 11:01:32+00:00,34,10-Q,001-36743,,5191740,1,1,aapl-20251227.htm,10-Q,2025'Q4
2,0000320193-25-000073,2025-08-01,2025-06-28,2025-08-01 10:00:42+00:00,34,10-Q,001-36743,,5304776,1,1,aapl-20250628.htm,10-Q,2025'Q2
3,0000320193-25-000057,2025-05-02,2025-03-29,2025-05-02 10:00:46+00:00,34,10-Q,001-36743,,5299807,1,1,aapl-20250329.htm,10-Q,2025'Q1
4,0000320193-25-000008,2025-01-31,2024-12-28,2025-01-31 11:01:27+00:00,34,10-Q,001-36743,,5150277,1,1,aapl-20241228.htm,10-Q,2024'Q4
5,0000320193-24-000081,2024-08-02,2024-06-29,2024-08-01 22:03:34+00:00,34,10-Q,001-36743,,5372771,1,1,aapl-20240629.htm,10-Q,2024'Q2
6,0000320193-24-000069,2024-05-03,2024-03-30,2024-05-02 22:04:25+00:00,34,10-Q,001-36743,,5284139,1,1,aapl-20240330.htm,10-Q,2024'Q1
7,0000320193-24-000006,2024-02-02,2023-12-30,2024-02-01 23:03:38+00:00,34,10-Q,001-36743,,4984121,1,1,aapl-20231230.htm,10-Q,2023'Q4
8,0000320193-23-000077,2023-08-04,2023-07-01,2023-08-03 22:04:43+00:00,34,10-Q,001-36743,,5939898,1,1,aapl-20230701.htm,10-Q,2023'Q3
9,0000320193-23-000064,2023-05-05,2023-04-01,2023-05-04 22:03:52+00:00,34,10-Q,001-36743,,6314786,1,1,aapl-20230401.htm,10-Q,2023'Q2


In [8]:
def fetching_reportings(ticker):
    all_dfs = []

    ticker_var = Company(ticker)
    ticker_cik = ticker_var.cik
    files = ticker_var.get_filings(form=["10-Q", "10-K"])

    for filing in files:
        try:
            xbrl = filing.xbrl()
            if not xbrl:
                continue
                
            income_statement = xbrl.statements.income_statement()
            if income_statement is None:
                continue

            df = income_statement.to_dataframe()
            df = df[df['abstract'] == False].copy()
            
            date_cols = [col for col in df.columns if col[0].isdigit()]
            df = df[['label'] + date_cols]
            df['label'] = df['label'].str.strip().str.replace(':', '')

            df = df.drop_duplicates(subset='label', keep='first')
            df = df.set_index('label')

            df_T = df.T
            df_T.index.name = 'period'
            df_T = df_T.reset_index()
            
            df_T['date'] = pd.to_datetime(df_T['period'].str.extract(r'(\d{4}-\d{2}-\d{2})')[0])
            df_T['period_type'] = df_T['period'].str.extract(r'\((\w+)\)')[0]

            # ---- CHANGE IS HERE ----
            df_T['period_type'] = df_T['period_type'].fillna(
                'Q4' if filing.form == '10-K' else 'SKIP'
            )
            # ------------------------

            df_T = df_T[df_T['period_type'].isin(['Q1', 'Q2', 'Q3', 'Q4'])]
            df_T = df_T.drop(columns=['period', 'period_type']).set_index('date')
            df_T = df_T.apply(pd.to_numeric, errors='coerce')
            
            all_dfs.append(df_T)

        except Exception as e:
            print(f"Failed processing {filing.company} ({filing.filing_date}): {e}")

    df_final = pd.concat(all_dfs)
    df_final['ticker'] = ticker
    df_final = df_final.loc[:, ~df_final.columns.duplicated(keep='first')]
    df_final = df_final[~df_final.index.duplicated(keep='first')]
    df_final = df_final.sort_index()

    df_final.columns = df_final.columns.str.lower().str.replace(' ', '_')
    return df_final

def calculating_margins(data): 
    df_margins = None 
    
    try:
        margins = ['net_sales', 'gross_margin', 'net_income']
        
        if set(margins).issubset(data.columns):
            
            df_margins = data[margins].copy()

            df_margins['gross_margin_rate'] = (df_margins['gross_margin'] / df_margins['net_sales']) * 100
            df_margins['net_margin_rate'] = (df_margins['net_income'] / df_margins['net_sales']) * 100

            df_margins.dropna(axis=0, inplace=True)
            
            df_margins['bps_difference'] = (df_margins['gross_margin_rate'] - df_margins['net_margin_rate']) 
            df_margins['dollar_difference'] = df_margins['gross_margin'] - df_margins['net_income'] 
        else:
            print("Yikes. One or more required columns are missing from the DataFrame.") 
            
    # Catching generic Exceptions ensures you see math/key errors, not just NameErrors
    except Exception as e:
        print(f"Error during calculation: {e}")

    return df_margins



In [10]:
fetching_reportings('WMT')


Filing(company='Walmart Inc.', cik=104169, form='10-Q/A', filing_date='2013-10-21', accession_no='0000104169-13-000039')
is an amended filing and may not contain full XBRL data e.g. some statements might be missing.
Consider using the original filing instead if available with `get_filings(form="10-K", amendments=False)`

No XBRL attachments found in filing Filing(company='Walmart Inc.', cik=104169, form='10-Q', filing_date='2009-06-05', accession_no='0000104169-09-000008')
No XBRL attachments found in filing Filing(company='Walmart Inc.', cik=104169, form='10-K', filing_date='2009-04-01', accession_no='0000104169-09-000006')
No XBRL attachments found in filing Filing(company='Walmart Inc.', cik=104169, form='10-Q', filing_date='2008-12-09', accession_no='0001193125-08-250391')
No XBRL attachments found in filing Filing(company='Walmart Inc.', cik=104169, form='10-Q', filing_date='2008-09-04', accession_no='0000104169-08-000006')
No XBRL attachments found in filing Filing(company='Walm

label,net_sales,operating_segments_-_walmart u.s.,operating_segments_-_walmart_international,operating_segments_-_sam's club_u.s.,grocery_-_walmart u.s.,general_merchandise_-_walmart u.s.,health_and_wellness_-_walmart u.s.,other_-_walmart u.s.,walmart u.s.,e_commerce_-_walmart u.s.,...,basic_income_per_share_from_continuing_operations_attributable_to_walmart,basic_income_per_share_from_discontinued_operations_attributable_to_walmart,basic_net_income_per_share_attributable_to_walmart,diluted_income_per_share_from_continuing_operations_attributable_to_walmart,diluted_income_(loss)_per_share_from_discontinued_operations_attributable_to_walmart,diluted_net_income_per_share_attributable_to_walmart,dividends_per_share,basic_(loss)_income_per_share_from_discontinued_operations_attributable_to_walmart,diluted_income_per_share_from_discontinued_operations_attributable_to_walmart,ticker
date,,,,,,,,,,,,,,,,,,,,,
2008-07-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.86,NaN,0.87,0.86,NaN,0.87,0.0,0.01,0.01,WMT
2008-10-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.77,0.03,0.80,0.77,0.03,0.80,0.0,NaN,NaN,WMT
2009-04-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,WMT
2009-07-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,WMT
2009-10-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,WMT
2010-04-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,WMT
2010-07-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,WMT
2010-10-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,WMT
2011-04-30,1.034150e+11,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,WMT


In [14]:
tickers = ['AAPL', 'nvda', 'googl', 'wmt', 'dis', 'v', 'acn', 'cof', 'ko']
tickers = [ticker.upper() for ticker in tickers]

all_raw_financials = []
all_calculated_margins = []

for ticker in tickers:
    print(f"Processing {ticker}...")
    financial_reports = fetching_reportings(ticker)
    
    if not financial_reports.empty:
        all_raw_financials.append(financial_reports)
        
        margins_for_company = calculating_margins(financial_reports)
        if margins_for_company is not None:
            # Tagging the dataframe so you know which company it belongs to
            margins_for_company['ticker'] = ticker
            all_calculated_margins.append(margins_for_company)

# Combine everything into clean master DataFrames if needed
if all_calculated_margins:
    df_margins_final = pd.concat(all_calculated_margins)
    print("\nProcessing complete! Sample data:")
    print(df_margins_final.head())

Processing AAPL...



Filing(company='Apple Inc.', cik=320193, form='10-K/A', filing_date='2010-01-25', accession_no='0001193125-10-012091')
is an amended filing and may not contain full XBRL data e.g. some statements might be missing.
Consider using the original filing instead if available with `get_filings(form="10-K", amendments=False)`


Filing(company='Apple Inc.', cik=320193, form='10-Q/A', filing_date='2009-04-27', accession_no='0001193125-09-087629')
is an amended filing and may not contain full XBRL data e.g. some statements might be missing.
Consider using the original filing instead if available with `get_filings(form="10-K", amendments=False)`

No XBRL attachments found in filing Filing(company='Apple Inc.', cik=320193, form='10-Q/A', filing_date='2009-04-27', accession_no='0001193125-09-087629')
No XBRL attachments found in filing Filing(company='Apple Inc.', cik=320193, form='10-Q', filing_date='2009-04-23', accession_no='0001193125-09-085781')
No XBRL attachments found in filing Filing(compa

Processing NVDA...


No XBRL attachments found in filing Filing(company='NVIDIA CORP', cik=1045810, form='10-Q', filing_date='2009-05-20', accession_no='0001045810-09-000017')
No XBRL attachments found in filing Filing(company='NVIDIA CORP', cik=1045810, form='10-K', filing_date='2009-03-13', accession_no='0001045810-09-000013')
No XBRL attachments found in filing Filing(company='NVIDIA CORP', cik=1045810, form='10-Q', filing_date='2008-12-02', accession_no='0001045810-08-000030')
No XBRL attachments found in filing Filing(company='NVIDIA CORP', cik=1045810, form='10-Q', filing_date='2008-08-21', accession_no='0001045810-08-000020')
No XBRL attachments found in filing Filing(company='NVIDIA CORP', cik=1045810, form='10-Q', filing_date='2008-05-22', accession_no='0001045810-08-000016')
No XBRL attachments found in filing Filing(company='NVIDIA CORP', cik=1045810, form='10-K', filing_date='2008-03-21', accession_no='0001045810-08-000011')
No XBRL attachments found in filing Filing(company='NVIDIA CORP', cik=

Yikes. One or more required columns are missing from the DataFrame.
Processing GOOGL...


...

Yikes. One or more required columns are missing from the DataFrame.
Processing COF...



Filing(company='CAPITAL ONE FINANCIAL CORP', cik=927628, form='10-K/A', filing_date='2011-03-07', accession_no='0001140361-11-014724')
is an amended filing and may not contain full XBRL data e.g. some statements might be missing.
Consider using the original filing instead if available with `get_filings(form="10-K", amendments=False)`

No XBRL attachments found in filing Filing(company='CAPITAL ONE FINANCIAL CORP', cik=927628, form='10-K', filing_date='2011-03-01', accession_no='0001140361-11-012916')
No XBRL attachments found in filing Filing(company='CAPITAL ONE FINANCIAL CORP', cik=927628, form='10-Q', filing_date='2009-05-08', accession_no='0001193125-09-105327')
No XBRL attachments found in filing Filing(company='CAPITAL ONE FINANCIAL CORP', cik=927628, form='10-K', filing_date='2009-02-26', accession_no='0001193125-09-039271')
No XBRL attachments found in filing Filing(company='CAPITAL ONE FINANCIAL CORP', cik=927628, form='10-Q', filing_date='2008-11-10', accession_no='000119312

Yikes. One or more required columns are missing from the DataFrame.
Processing KO...



Filing(company='COCA COLA CO', cik=21344, form='10-Q/A', filing_date='2024-05-30', accession_no='0000021344-24-000019')
is an amended filing and may not contain full XBRL data e.g. some statements might be missing.
Consider using the original filing instead if available with `get_filings(form="10-K", amendments=False)`

Failed to resolve IncomeStatement for COCA COLA CO (CIK: Unknown, Period: 2024-03-29). No matching statements found. No statements available. No statements available in XBRL data

Filing(company='COCA COLA CO', cik=21344, form='10-Q/A', filing_date='2014-04-30', accession_no='0000021344-14-000018')
is an amended filing and may not contain full XBRL data e.g. some statements might be missing.
Consider using the original filing instead if available with `get_filings(form="10-K", amendments=False)`

No XBRL attachments found in filing Filing(company='COCA COLA CO', cik=21344, form='10-Q/A', filing_date='2014-04-30', accession_no='0000021344-14-000018')

Filing(company='CO

Yikes. One or more required columns are missing from the DataFrame.

Processing complete! Sample data:
label          net_sales  gross_margin    net_income  gross_margin_rate  \
date                                                                      
2009-06-27  9.734000e+09  3.983000e+09  1.828000e+09          40.918430   
2009-12-26  1.568300e+10  6.411000e+09  3.378000e+09          40.878658   
2010-03-27  1.349900e+10  5.625000e+09  3.074000e+09          41.669753   
2010-06-26  1.570000e+10  6.136000e+09  3.253000e+09          39.082803   
2010-12-25  2.674100e+10  1.029800e+10  6.004000e+09          38.510153   

label       net_margin_rate  bps_difference  dollar_difference ticker  
date                                                                   
2009-06-27        18.779536       22.138895       2.155000e+09   AAPL  
2009-12-26        21.539246       19.339412       3.033000e+09   AAPL  
2010-03-27        22.772057       18.897696       2.551000e+09   AAPL  
2010-06-26 

In [21]:
all_calculated_margins[0]

for item in all_calculated_margins:
    print(item)

label          net_sales  gross_margin    net_income  gross_margin_rate  \
date                                                                      
2009-06-27  9.734000e+09  3.983000e+09  1.828000e+09          40.918430   
2009-12-26  1.568300e+10  6.411000e+09  3.378000e+09          40.878658   
2010-03-27  1.349900e+10  5.625000e+09  3.074000e+09          41.669753   
2010-06-26  1.570000e+10  6.136000e+09  3.253000e+09          39.082803   
2010-12-25  2.674100e+10  1.029800e+10  6.004000e+09          38.510153   
2011-03-26  2.466700e+10  1.021800e+10  5.987000e+09          41.423765   
2011-06-25  2.857100e+10  1.192200e+10  7.308000e+09          41.727626   
2011-12-31  4.633300e+10  2.070300e+10  1.306400e+10          44.683055   
2012-03-31  3.918600e+10  1.856400e+10  1.162200e+10          47.374062   
2012-06-30  3.502300e+10  1.499400e+10  8.824000e+09          42.811866   
2012-12-29  5.451200e+10  2.106000e+10  1.307800e+10          38.633695   
2013-03-30  4.360300e+10 

In [ ]:
# import matplotlib.pyplot as plt

# plt.figure(figsize=(20, 8))

# df_margins.plot(y='gross_margin', ax=plt.gca(), linewidth=2)
# plt.title('Gross Margin Evolution Across Financial Quarters', fontsize=16, fontweight='bold', pad=15)
# plt.xlabel('Report Date / Quarter', fontsize=12)
# plt.ylabel('Gross Margin Value', fontsize=12)
# plt.grid(True, linestyle='--', alpha=0.6) # Adds a light background grid
# plt.tight_layout()
# plt.show()

# df_margins.plot(y ='net_margin')